# 02. Forward Kinematics In Pinocchio

This notebook focuses on the most important practical distinction in Pinocchio: the `model` stores the robot structure, while the `data` stores the results of computations such as forward kinematics.

## What you should remember
- `model` is structural and persistent; `data` is the workspace that gets updated by algorithms.
- `pin.forwardKinematics(model, data, q)` updates joint placements in `data.oMi`.
- `pin.updateFramePlacements(model, data)` updates frame placements in `data.oMf` after joint kinematics are known.
- `pin.framesForwardKinematics(model, data, q)` is a convenient bulk call when you want all frame placements at once.
- Use joint IDs for `data.oMi` and frame IDs for `data.oMf`.

In [ ]:
%%capture
!pip install pin viser robot_descriptions numpy scipy matplotlib trimesh

import site
site.main()

In [ ]:
%%capture
!wget -O viz.py https://raw.githubusercontent.com/Atarilab/colab_utils/refs/heads/main/viz.py
import viz

If the next cell gives an error restart the session to load the libraries (ctrl + m + .) or click runtime -> restart session. Then rerun the second and third codeblocks (do not rerun the first block!).

In [ ]:
import numpy as np
import pinocchio as pin
from robot_descriptions.loaders.pinocchio import load_robot_description

from viz import PinNotebookViz, create_server

server, share_url = create_server()
notebook_viz = PinNotebookViz(server)

print(f"Open the visualizer here: {share_url}")

In [ ]:
robot = load_robot_description("ur5_description")
model = robot.model
data = model.createData()

notebook_viz.attach_robot(robot)
q0 = pin.neutral(model)
notebook_viz.display(q0)

print(f"nq = {model.nq}, nv = {model.nv}")
print(f"model type: {type(model)}")
print(f"data type:  {type(data)}")

## Notation box
- `o` means the world or origin frame.
- `i` is a joint index.
- `f` is a frame index.
- `oMi` is the transform from the world frame to joint `i`.
- `oMf` is the transform from the world frame to frame `f`.

For a fixed-base arm like UR5, `nq == nv`, but this is **not** true in general, especially for floating-base systems.

## `model` versus `data`
Before any kinematics call, `data` does not yet contain the placements for the configuration we care about.

A very common mistake is to forget that `data` is mutable. If you modify `q` and then read `data.oMi` or `data.oMf` without recomputing forward kinematics, you are reading old results.

In [ ]:
base_joint_id = model.getJointId("shoulder_pan_joint")
tool_frame_id = model.getFrameId("tool0")

print("Joint id for shoulder_pan_joint:", base_joint_id)
print("Frame id for tool0:         ", tool_frame_id)
print()
print("Joint placement before FK:")
print(data.oMi[base_joint_id])
print()
print("Frame placement before FK:")
print(data.oMf[tool_frame_id])


## Query joints with `data.oMi`
`pin.forwardKinematics(...)` updates the joint placements. These are stored in `data.oMi`.

In [ ]:
q = np.array([0.2, -1.1, 1.2, -0.7, 0.5, 0.3])
pin.forwardKinematics(model, data, q)

wrist_joint_id = model.getJointId("wrist_3_joint")
print("Placement of wrist_3_joint in the world frame:")
print(data.oMi[wrist_joint_id])

## Query frames with `data.oMf`
Frame placements live in `data.oMf`. A robot often has useful frames attached to links, tools, sensors, or fingertips that are not joints themselves, so Pinocchio separates joint placements (`oMi`) from frame placements (`oMf`). For example, `tool0` is a frame, not a joint.

In [ ]:
pin.updateFramePlacements(model, data)
base_frame_id = model.getFrameId("base_link")
tool_frame_id = model.getFrameId("tool0")

oMbase = data.oMf[base_frame_id]
oMtool = data.oMf[tool_frame_id]

print("Base frame translation:", oMbase.translation)
print("Tool frame translation:", oMtool.translation)

notebook_viz.clear_frames()
notebook_viz.display(q)
notebook_viz.show_frame("base_link", oMbase)
notebook_viz.show_frame("tool0", oMtool)

## Bulk frame updates versus one-frame updates
If you want many frame placements, use `pin.framesForwardKinematics(...)` once.

It is equivalent to calling:
```python
pin.forwardKinematics(model, data, q)
pin.updateFramePlacements(model, data)
```

If you only need one particular frame after joint kinematics are known, `pin.updateFramePlacement(...)` can be more targeted.

In [ ]:
q_random = pin.randomConfiguration(model)

pin.framesForwardKinematics(model, data, q_random)
bulk_tool = pin.SE3(data.oMf[tool_frame_id].rotation.copy(), data.oMf[tool_frame_id].translation.copy())

pin.forwardKinematics(model, data, q_random)
pin.updateFramePlacements(model, data)
split_tool = pin.SE3(data.oMf[tool_frame_id].rotation.copy(), data.oMf[tool_frame_id].translation.copy())

pin.forwardKinematics(model, data, q_random)
single_tool = pin.updateFramePlacement(model, data, tool_frame_id)

print("Difference between bulk and split update:", np.linalg.norm(bulk_tool.translation - split_tool.translation))
print("Difference between bulk and single-frame update:", np.linalg.norm(bulk_tool.translation - single_tool.translation))

## Common mistake: stale data after changing `q`
Once `q` changes, previously stored placements are stale. Recompute kinematics before trusting `data.oMi` or `data.oMf` again.

In [ ]:
stale_translation = data.oMf[tool_frame_id].translation.copy()
q_new = pin.randomConfiguration(model)

print("Tool translation stored from the previous configuration:", stale_translation)
print("After changing q but before recomputing FK, data.oMf still contains:")
print(data.oMf[tool_frame_id].translation)

pin.framesForwardKinematics(model, data, q_new)
print("After recomputing FK, the tool translation becomes:")
print(data.oMf[tool_frame_id].translation)

## Relative transforms are often what you really need
Once the placements are updated, you can combine them with ordinary `SE3` algebra.

In [ ]:
pin.framesForwardKinematics(model, data, q)
oMbase = data.oMf[base_frame_id]
oMtool = data.oMf[tool_frame_id]
baseMtool = oMbase.inverse() * oMtool

print("Transform from base_link to tool0:")
print(baseMtool)

## Exercise
Compute `toolMbase` and verify that `baseMtool * toolMbase` is the identity transform.

Try it yourself first, then run the check cell.

In [ ]:
pin.framesForwardKinematics(model, data, q)
oMbase = data.oMf[base_frame_id]
oMtool = data.oMf[tool_frame_id]
baseMtool = oMbase.inverse() * oMtool
toolMbase = oMtool.inverse() * oMbase
identity_error = np.linalg.norm((baseMtool * toolMbase).homogeneous - np.eye(4))

print("toolMbase =")
print(toolMbase)
print()
print("Identity error:", identity_error)
